# Notebook 09 — Lag Circular Correlation Between Three 2D Representations
**Project:** ENSO-BSISO Self-Supervised Learning  
**Author:** Jiayi (jh9141@nyu.edu)

Implements **Plan 2** from Session 14: pairwise lag circular correlation between the three
2D representations of BSISO state.

## Three representations

| ID | Object | Source | N days |
|----|--------|--------|--------|
| `idx` | BSISO index (conventional) | APEC `BSISO.INDEX.NORM.LY.data` — PC1/PC2 re-parsed directly | 6,579 |
| `sup` | Supervised 2D encoder (nb 07c, 128-layer, no L2) | `results/lee_2d_no_l2/embeddings.npy` | 6,579 |
| `ssl` | SSL temporal 2D encoder (nb 08, 32-layer, no L2) | `results/lee_2d_ssl/embeddings.npy` | 4,429 |

> **Note on `idx`:** `labels_aligned_mjjas_lee.csv` only stores discrete phase (1–8), not continuous
> PC1/PC2. Cell 2 re-parses the raw BSISO file to recover the continuous angle θ = atan2(PC2, PC1).

## Scalar quantity: angle θ = atan2(z₂, z₁)

Each 2D representation lies in R², so the angle θ captures its position on the BSISO cycle.
For the BSISO index: θ = atan2(PC2, PC1). For learned embeddings: θ = atan2(emb[:,1], emb[:,0]).

## Correlation method: circular correlation coefficient (Jammalamadaka & SenGupta 2001)

```
ρ_c(θ₁, θ₂) = Σ sin(θ₁ᵢ − θ̄₁) sin(θ₂ᵢ − θ̄₂)
               ─────────────────────────────────────
               √[Σ sin²(θ₁ᵢ − θ̄₁) · Σ sin²(θ₂ᵢ − θ̄₂)]
```

θ̄ = circular mean = atan2(mean sin θ, mean cos θ).  
Range: [−1, +1]. **Invariant to constant rotation** of either variable — so no Procrustes
alignment between representations is needed.

## Three pairwise lag correlations (τ ∈ [−30, +30] days)

- `ρ_c(idx, sup; τ)` — BSISO index vs. supervised embedding
- `ρ_c(idx, ssl; τ)` — BSISO index vs. SSL embedding
- `ρ_c(sup, ssl; τ)` — supervised vs. SSL embedding

Convention: **τ > 0 means A leads B** (A(d) correlated with B(d+τ)).  
Pairs are formed only within the same calendar year (no May-Sep cross-year bleeding).

---

## Cell 1 — Mount Drive + Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR   = '/content/drive/MyDrive/BSISO_SSL_Project'
PROCESSED_DIR = f'{PROJECT_DIR}/data/processed'
RESULTS_DIR   = f'{PROJECT_DIR}/results'
OUT_DIR       = f'{RESULTS_DIR}/lag_correlation'

os.makedirs(OUT_DIR, exist_ok=True)

# Embedding files
SUP_EMB_FILE    = f'{RESULTS_DIR}/lee_2d_no_l2/embeddings.npy'   # 128-layer supervised
SSL_EMB_FILE    = f'{RESULTS_DIR}/lee_2d_ssl/embeddings.npy'     # 32-layer SSL

# Label / date files
LABELS_SUP_FILE = f'{PROCESSED_DIR}/labels_aligned_mjjas_lee.csv'     # sup + idx dates (6579)
LABELS_SSL_FILE = f'{PROCESSED_DIR}/labels_aligned_mjjas_lee_lp25.csv' # ssl dates (4429)

for f in [SUP_EMB_FILE, SSL_EMB_FILE, LABELS_SUP_FILE, LABELS_SSL_FILE]:
    exists = os.path.exists(f)
    print(f'  {"OK" if exists else "MISSING"}: {os.path.basename(f)}')
print('Drive mounted.')

## Cell 2 — Load All Three Representations

In [ ]:
import numpy as np
import pandas as pd

# --- Aligned labels (dates + phase/ENSO only — PC1/PC2 were dropped when saving) ---
df_sup = pd.read_csv(LABELS_SUP_FILE, parse_dates=['date'])
dates_sup = pd.DatetimeIndex(df_sup['date'].values)

df_ssl = pd.read_csv(LABELS_SSL_FILE, parse_dates=['date'])
dates_ssl = pd.DatetimeIndex(df_ssl['date'].values)

# --- Load embeddings ---
emb_sup = np.load(SUP_EMB_FILE)  # (6579, 2)
emb_ssl = np.load(SSL_EMB_FILE)  # (4429, 2)

assert len(emb_sup) == len(dates_sup), f'sup mismatch: {len(emb_sup)} vs {len(dates_sup)}'
assert len(emb_ssl) == len(dates_ssl), f'ssl mismatch: {len(emb_ssl)} vs {len(dates_ssl)}'

# --- BSISO index angle: re-parse raw file to get continuous PC1/PC2 ---
# labels_mjjas.csv only saves discrete phase (1-8), not the raw PC components.
# Re-parse BSISO.INDEX.NORM.LY.data exactly as notebook 02 does.
BSISO_RAW_FILE = f'{PROJECT_DIR}/data/raw/BSISO.INDEX.NORM.LY.data'

rows = []
with open(BSISO_RAW_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        if not line or line[0].isalpha() or line.startswith('#'):
            continue
        parts = line.split()
        if len(parts) >= 7:
            rows.append(parts)

col_names = ['year', 'doy', 'pc1_bsiso1', 'pc2_bsiso1',
             'pc1_bsiso2', 'pc2_bsiso2', 'bsiso1_amp', 'bsiso2_amp']
df_bsiso = pd.DataFrame(rows, columns=col_names[:len(rows[0])])
df_bsiso['year']       = df_bsiso['year'].astype(int)
df_bsiso['doy']        = df_bsiso['doy'].astype(int)
df_bsiso['pc1_bsiso1'] = df_bsiso['pc1_bsiso1'].astype(float)
df_bsiso['pc2_bsiso1'] = df_bsiso['pc2_bsiso1'].astype(float)
df_bsiso['date'] = pd.to_datetime(
    df_bsiso['year'].astype(str) + df_bsiso['doy'].astype(str).str.zfill(3),
    format='%Y%j'
).dt.normalize()

# Merge continuous PC1/PC2 onto aligned dates
df_merged = df_sup[['date']].merge(
    df_bsiso[['date', 'pc1_bsiso1', 'pc2_bsiso1']],
    on='date', how='left'
)
n_missing = df_merged['pc1_bsiso1'].isna().sum()
print(f'BSISO raw file: {len(df_bsiso)} records parsed')
print(f'PC1/PC2 merge onto {len(df_merged)} aligned dates: {n_missing} missing')

pc1 = df_merged['pc1_bsiso1'].values.astype(float)
pc2 = df_merged['pc2_bsiso1'].values.astype(float)
theta_idx = np.arctan2(pc2, pc1)   # continuous angle in (-π, π]

# --- Supervised 2D: atan2(z2, z1) ---
theta_sup = np.arctan2(emb_sup[:, 1], emb_sup[:, 0])

# --- SSL 2D: atan2(z2, z1) ---
theta_ssl = np.arctan2(emb_ssl[:, 1], emb_ssl[:, 0])

print(f'\nidx: {len(theta_idx)} days, {dates_sup[0].date()} → {dates_sup[-1].date()}')
print(f'sup: {len(theta_sup)} days, {dates_sup[0].date()} → {dates_sup[-1].date()}')
print(f'ssl: {len(theta_ssl)} days, {dates_ssl[0].date()} → {dates_ssl[-1].date()}')

print('\nAngle ranges (rad):')
for name, th in [('idx', theta_idx), ('sup', theta_sup), ('ssl', theta_ssl)]:
    cm = np.degrees(np.arctan2(np.mean(np.sin(th)), np.mean(np.cos(th))))
    print(f'  {name}: [{th.min():.2f}, {th.max():.2f}]  circular mean = {cm:.1f}°')

## Cell 3 — Circular Correlation Function + Lag Computation

**Jammalamadaka & SenGupta (2001) circular correlation:**

$$\rho_c(\theta_1, \theta_2) = \frac{\sum \sin(\theta_{1i} - \bar{\theta}_1) \sin(\theta_{2i} - \bar{\theta}_2)}{\sqrt{\sum \sin^2(\theta_{1i} - \bar{\theta}_1) \cdot \sum \sin^2(\theta_{2i} - \bar{\theta}_2)}}$$

This is **rotation-invariant**: if θ₂ = θ₁ + constant, ρ_c = 1. No Procrustes alignment needed.

In [ ]:
import numpy as np
import pandas as pd

def circular_mean(theta):
    return np.arctan2(np.mean(np.sin(theta)), np.mean(np.cos(theta)))

def circular_corr(theta1, theta2):
    """Jammalamadaka & SenGupta circular correlation coefficient."""
    if len(theta1) < 5:
        return np.nan
    mu1 = circular_mean(theta1)
    mu2 = circular_mean(theta2)
    s1 = np.sin(theta1 - mu1)
    s2 = np.sin(theta2 - mu2)
    num = np.sum(s1 * s2)
    den = np.sqrt(np.sum(s1**2) * np.sum(s2**2))
    return num / den if den > 0 else 0.0

def lag_circular_corr(theta_A, dates_A, theta_B, dates_B,
                      max_lag=30, min_pairs=30):
    """
    Compute circular lag correlation ρ_c(A, B; τ) for τ in [-max_lag, +max_lag].
    Convention: τ > 0 means A leads B — correlates A(d) with B(d+τ).
    Pairs are valid only when d and d+τ share the same calendar year.
    Returns: lags, rho (array), n_pairs (array)
    """
    # Build date → index lookup for B
    date_to_idx_B = {d: i for i, d in enumerate(dates_B)}

    lags    = np.arange(-max_lag, max_lag + 1)
    rho     = np.full(len(lags), np.nan)
    n_pairs = np.zeros(len(lags), dtype=int)

    for k, tau in enumerate(lags):
        a_vals, b_vals = [], []
        for i, d in enumerate(dates_A):
            d_shifted = d + pd.Timedelta(days=int(tau))
            if d_shifted.year != d.year:      # no cross-year pairs
                continue
            if d_shifted in date_to_idx_B:
                j = date_to_idx_B[d_shifted]
                a_vals.append(theta_A[i])
                b_vals.append(theta_B[j])

        n_pairs[k] = len(a_vals)
        if n_pairs[k] >= min_pairs:
            rho[k] = circular_corr(np.array(a_vals), np.array(b_vals))

    return lags, rho, n_pairs

print('Functions defined.')
print(f'Circular correlation range: [-1, 1], rotation-invariant.')

## Cell 4 — Compute All Three Lag Correlations

In [ ]:
MAX_LAG = 30

print('Computing ρ_c(idx, sup; τ)  ...')
lags, rho_idx_sup, n_idx_sup = lag_circular_corr(
    theta_idx, dates_sup,
    theta_sup, dates_sup,
    max_lag=MAX_LAG
)
print(f'  peak ρ = {np.nanmax(rho_idx_sup):.3f} at τ = {lags[np.nanargmax(rho_idx_sup)]} days')
print(f'  ρ at τ=0: {rho_idx_sup[lags==0][0]:.3f}')

print('Computing ρ_c(idx, ssl; τ)  ...')
lags, rho_idx_ssl, n_idx_ssl = lag_circular_corr(
    theta_idx, dates_sup,   # idx shares sup dates
    theta_ssl, dates_ssl,
    max_lag=MAX_LAG
)
print(f'  peak ρ = {np.nanmax(rho_idx_ssl):.3f} at τ = {lags[np.nanargmax(rho_idx_ssl)]} days')
print(f'  ρ at τ=0: {rho_idx_ssl[lags==0][0]:.3f}')

print('Computing ρ_c(sup, ssl; τ)  ...')
lags, rho_sup_ssl, n_sup_ssl = lag_circular_corr(
    theta_sup, dates_sup,
    theta_ssl, dates_ssl,
    max_lag=MAX_LAG
)
print(f'  peak ρ = {np.nanmax(rho_sup_ssl):.3f} at τ = {lags[np.nanargmax(rho_sup_ssl)]} days')
print(f'  ρ at τ=0: {rho_sup_ssl[lags==0][0]:.3f}')

print('\nPair counts at τ=0:')
print(f'  idx-sup: {n_idx_sup[lags==0][0]}')
print(f'  idx-ssl: {n_idx_ssl[lags==0][0]}')
print(f'  sup-ssl: {n_sup_ssl[lags==0][0]}')

## Cell 5 — Permutation Significance Bands

For each pair at each lag: shuffle the B-side angles (within-year) and recompute ρ_c.
Report the 95th percentile of the null distribution as the significance threshold.

In [ ]:
import numpy as np

N_PERM = 500  # permutation draws per pair; increase to 1000 for final publication
RNG = np.random.default_rng(42)

def permutation_null_band(theta_A, dates_A, theta_B, dates_B,
                           max_lag=30, n_perm=500, quantile=0.95):
    """
    Within-year permutation null for lag circular correlation.
    Shuffles B angles independently within each year, recomputes ρ_c for all lags.
    Returns the (quantile) across all permutations and all lags (conservative upper bound).
    """
    # Group B indices by year
    years_B = pd.DatetimeIndex(dates_B).year
    year_groups_B = {}
    for i, y in enumerate(years_B):
        year_groups_B.setdefault(y, []).append(i)

    null_rhos = []  # collect all null ρ values

    for _ in range(n_perm):
        # Shuffle B within each year
        theta_B_perm = theta_B.copy()
        for idx_list in year_groups_B.values():
            perm = RNG.permutation(len(idx_list))
            theta_B_perm[idx_list] = theta_B[np.array(idx_list)[perm]]

        _, rho_perm, _ = lag_circular_corr(
            theta_A, dates_A, theta_B_perm, dates_B, max_lag=max_lag
        )
        null_rhos.extend(np.abs(rho_perm[~np.isnan(rho_perm)]))

    return float(np.quantile(null_rhos, quantile))

print('Computing permutation null bands (500 permutations × 3 pairs)...')
print('  ρ_c(idx, sup) null band...')
null_idx_sup = permutation_null_band(theta_idx, dates_sup, theta_sup, dates_sup,
                                      max_lag=MAX_LAG, n_perm=N_PERM)
print(f'    95th pct null |ρ| = {null_idx_sup:.3f}')

print('  ρ_c(idx, ssl) null band...')
null_idx_ssl = permutation_null_band(theta_idx, dates_sup, theta_ssl, dates_ssl,
                                      max_lag=MAX_LAG, n_perm=N_PERM)
print(f'    95th pct null |ρ| = {null_idx_ssl:.3f}')

print('  ρ_c(sup, ssl) null band...')
null_sup_ssl = permutation_null_band(theta_sup, dates_sup, theta_ssl, dates_ssl,
                                      max_lag=MAX_LAG, n_perm=N_PERM)
print(f'    95th pct null |ρ| = {null_sup_ssl:.3f}')
print('Done.')

## Cell 6 — Plot: Three-Panel Lag Correlation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig.suptitle('Pairwise Lag Circular Correlation  ρ_c(A, B; τ)\n'
             'τ > 0: A leads B  |  within-year pairs  |  θ = atan2(z₂, z₁)',
             fontsize=13, fontweight='bold')

panels = [
    (rho_idx_sup, null_idx_sup, n_idx_sup, 'ρ_c(BSISO index,  Supervised 2D)',   '#1f77b4'),
    (rho_idx_ssl, null_idx_ssl, n_idx_ssl, 'ρ_c(BSISO index,  SSL 2D)',           '#2ca02c'),
    (rho_sup_ssl, null_sup_ssl, n_sup_ssl, 'ρ_c(Supervised 2D,  SSL 2D)',         '#d62728'),
]

for ax, (rho, null_band, npairs, title, color) in zip(axes, panels):
    # Correlation curve
    ax.plot(lags, rho, color=color, lw=2, zorder=3)
    ax.fill_between(lags, rho, 0, where=np.abs(rho) > null_band,
                    color=color, alpha=0.25, zorder=2, label='significant')

    # Significance band
    ax.axhline( null_band, color='gray', ls='--', lw=1, alpha=0.7, label=f'95% null ({null_band:.3f})')
    ax.axhline(-null_band, color='gray', ls='--', lw=1, alpha=0.7)

    # Reference lines
    ax.axhline(0, color='black', lw=0.8)
    ax.axvline(0, color='black', lw=0.8, ls=':')

    # Peak annotation
    peak_idx = np.nanargmax(rho)
    peak_tau = lags[peak_idx]
    peak_rho = rho[peak_idx]
    ax.annotate(f'peak τ={peak_tau:+d}d\nρ={peak_rho:.3f}',
                xy=(peak_tau, peak_rho),
                xytext=(peak_tau + 5 * np.sign(peak_tau - MAX_LAG/2), peak_rho - 0.05),
                fontsize=9, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Lag τ (days)', fontsize=10)
    ax.set_ylabel('ρ_c (circular correlation)', fontsize=10)
    ax.set_xlim(-MAX_LAG, MAX_LAG)
    ax.set_ylim(-0.5, 1.0)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)

    # Secondary x-axis: pair count
    ax2 = ax.twinx()
    ax2.plot(lags, npairs, color='orange', lw=1, alpha=0.5, ls='-')
    ax2.set_ylabel('N pairs', color='orange', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='orange', labelsize=7)
    ax2.set_ylim(0, npairs.max() * 2)

plt.tight_layout()
out_path = f'{OUT_DIR}/lag_circular_corr.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## Cell 7 — Overlay Plot (All Three Curves Together)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

curves = [
    (rho_idx_sup, null_idx_sup, 'idx ↔ sup', '#1f77b4'),
    (rho_idx_ssl, null_idx_ssl, 'idx ↔ ssl', '#2ca02c'),
    (rho_sup_ssl, null_sup_ssl, 'sup ↔ ssl', '#d62728'),
]

for rho, null_band, label, color in curves:
    ax.plot(lags, rho, color=color, lw=2, label=label)
    ax.axhline(null_band, color=color, lw=0.8, ls=':', alpha=0.6)

ax.axhline(0, color='black', lw=0.8)
ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Lag τ (days)  [τ > 0: row representation leads]', fontsize=11)
ax.set_ylabel('ρ_c (circular correlation coefficient)', fontsize=11)
ax.set_title('Lag Circular Correlation — BSISO index / Supervised 2D / SSL 2D', fontsize=12)
ax.set_xlim(-MAX_LAG, MAX_LAG)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = f'{OUT_DIR}/lag_circular_corr_overlay.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## Cell 8 — Numerical Summary + Save CSV

In [ ]:
import pandas as pd
import numpy as np
import json

def peak_info(lags, rho, null_band, name):
    valid = ~np.isnan(rho)
    peak_idx  = np.nanargmax(rho)
    trough_idx = np.nanargmin(rho)
    tau0_rho  = rho[lags == 0][0]
    return {
        'pair'              : name,
        'rho_at_tau0'       : round(float(tau0_rho), 4),
        'peak_rho'          : round(float(rho[peak_idx]), 4),
        'peak_tau_days'     : int(lags[peak_idx]),
        'trough_rho'        : round(float(rho[trough_idx]), 4),
        'trough_tau_days'   : int(lags[trough_idx]),
        'null_95pct'        : round(null_band, 4),
        'n_sig_lags'        : int(np.sum(np.abs(rho[valid]) > null_band)),
        'n_total_lags'      : int(valid.sum()),
    }

summary = [
    peak_info(lags, rho_idx_sup, null_idx_sup, 'idx ↔ sup'),
    peak_info(lags, rho_idx_ssl, null_idx_ssl, 'idx ↔ ssl'),
    peak_info(lags, rho_sup_ssl, null_sup_ssl, 'sup ↔ ssl'),
]

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

# Save CSV
csv_path = f'{OUT_DIR}/lag_corr_summary.csv'
df_summary.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

# Save full curves
df_curves = pd.DataFrame({
    'lag'         : lags,
    'rho_idx_sup' : rho_idx_sup,
    'rho_idx_ssl' : rho_idx_ssl,
    'rho_sup_ssl' : rho_sup_ssl,
    'n_idx_sup'   : n_idx_sup,
    'n_idx_ssl'   : n_idx_ssl,
    'n_sup_ssl'   : n_sup_ssl,
})
curves_path = f'{OUT_DIR}/lag_corr_curves.csv'
df_curves.to_csv(curves_path, index=False)
print(f'Saved: {curves_path}')

## Cell 9 — Autocorrelation Check (Each Representation vs Itself)

The autocorrelation shape shows the intrinsic memory of each representation.
BSISO has a characteristic ~30-day period — the autocorrelation should show a
positive lobe (0–15 days) followed by a negative lobe (15–30 days, half-period phase flip).

In [ ]:
import matplotlib.pyplot as plt

print('Computing autocorrelations...')
_, acf_idx, _ = lag_circular_corr(theta_idx, dates_sup, theta_idx, dates_sup, max_lag=MAX_LAG)
_, acf_sup, _ = lag_circular_corr(theta_sup, dates_sup, theta_sup, dates_sup, max_lag=MAX_LAG)
_, acf_ssl, _ = lag_circular_corr(theta_ssl, dates_ssl, theta_ssl, dates_ssl, max_lag=MAX_LAG)

fig, ax = plt.subplots(figsize=(10, 4))

for acf, label, color in [
    (acf_idx, 'BSISO index (idx)', '#1f77b4'),
    (acf_sup, 'Supervised 2D (sup)', '#ff7f0e'),
    (acf_ssl, 'SSL 2D (ssl)',        '#2ca02c'),
]:
    ax.plot(lags, acf, color=color, lw=2, label=label)

ax.axhline(0, color='black', lw=0.8)
ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
ax.set_xlabel('Lag τ (days)', fontsize=11)
ax.set_ylabel('ρ_c (circular autocorrelation)', fontsize=11)
ax.set_title('Circular Autocorrelation — Intrinsic temporal memory of each representation',
             fontsize=11)
ax.set_xlim(0, MAX_LAG)  # only positive lags for autocorrelation
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = f'{OUT_DIR}/autocorrelation.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

# Print half-power decay point (lag where autocorrelation drops to 0.5)
for acf, name in [(acf_idx, 'idx'), (acf_sup, 'sup'), (acf_ssl, 'ssl')]:
    pos_lags = lags[lags >= 0]
    pos_acf  = acf[lags >= 0]
    below = pos_lags[pos_acf < 0.5]
    if len(below) > 0:
        print(f'  {name}: autocorr < 0.5 first at τ = {below[0]} days')
    else:
        print(f'  {name}: autocorr stays > 0.5 throughout [-30, +30]')

## Cell 10 — Summary Report

In [ ]:
import json
import numpy as np

report_lines = [
    '=' * 65,
    'PLAN 2 — Lag Circular Correlation Summary',
    f'Date: {pd.Timestamp.now().strftime("%Y-%m-%d")}',
    '=' * 65,
    '',
    'Scalar quantity: θ = atan2(z₂, z₁)  (angle in R²)',
    'Method: Jammalamadaka & SenGupta circular correlation ρ_c',
    'Convention: τ > 0 → A leads B',
    'Significance: within-year permutation null, 95th percentile',
    '',
    'PAIR SUMMARY',
    '-' * 65,
]

for row in summary:
    report_lines += [
        f"Pair: {row['pair']}",
        f"  ρ_c at τ=0:   {row['rho_at_tau0']:.4f}",
        f"  Peak ρ_c:     {row['peak_rho']:.4f}  at τ = {row['peak_tau_days']:+d} days",
        f"  Trough ρ_c:   {row['trough_rho']:.4f}  at τ = {row['trough_tau_days']:+d} days",
        f"  95% null:     {row['null_95pct']:.4f}",
        f"  Significant lags: {row['n_sig_lags']} / {row['n_total_lags']}",
        '',
    ]

report_lines += [
    'OUTPUT FILES',
    '-' * 65,
    f'  lag_circular_corr.png          — 3-panel lag correlation',
    f'  lag_circular_corr_overlay.png  — overlay of all three curves',
    f'  autocorrelation.png            — circular autocorrelation per repr.',
    f'  lag_corr_summary.csv           — peak/trough table',
    f'  lag_corr_curves.csv            — full ρ_c(τ) arrays',
    '=' * 65,
]

report_text = '\n'.join(report_lines)
print(report_text)

report_path = f'{OUT_DIR}/lag_corr_report.txt'
with open(report_path, 'w') as f:
    f.write(report_text)
print(f'\nSaved: {report_path}')

---
## Expected Outputs

```
BSISO_SSL_Project/results/lag_correlation/
  lag_circular_corr.png          ← 3-panel (one per pair), significance shading
  lag_circular_corr_overlay.png  ← all three curves on one axis
  autocorrelation.png            ← circular ACF for each representation
  lag_corr_summary.csv           ← peak τ, peak ρ, null threshold per pair
  lag_corr_curves.csv            ← full ρ_c(τ) arrays for all three pairs
  lag_corr_report.txt            ← plain-text summary
```

## What to look for

| What you observe | Interpretation |
|---|---|
| ρ_c(idx, sup; 0) high | Supervised embedding tracks BSISO index at same time |
| ρ_c(idx, sup) peaks at τ ≠ 0 | Supervised leads/lags BSISO index by that many days |
| ρ_c(idx, ssl) lower than ρ_c(idx, sup) | Expected — SSL not trained on BSISO labels |
| ρ_c(sup, ssl) moderate | Two learned representations share latent structure despite different training |
| Autocorr decay width | How many days of temporal memory each representation has (should reflect ~30-day BSISO cycle) |

---
*DDCS Project | jh9141@nyu.edu*